In [3]:
import textwrap

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage


from rag_pipeline import split_text_add_video_metadata, clean_classification_text
from video_processing import analyze_video

from chat_memory import get_chat_history, last_n_messages

from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

#### Jeff Nippard: Overhead Press transcript

In [4]:
jeff_ohp = "/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/processed/cleaned_nippard_ohp_dict"
jeff_ohp_chunked = split_text_add_video_metadata(jeff_ohp)
print("Chunks:", len(jeff_ohp_chunked))
print("Sample chunk:", jeff_ohp_chunked[:1])

Chunks: 11
Sample chunk: [Document(metadata={'video_id': '_RlRDWO2jfg', 'title': 'Build Bigger Shoulders With Perfect Training Technique (The Overhead Press)', 'author': 'Jeff Nippard', 'difficulty': 'intermediate', 'exercise_type': 'overhead_press'}, page_content="Okay, welcome everyone to a new episode of Technique Tuesday. This week, we're going to be looking at how to perform the overhead barbell press, or OHP, with perfect technique. With this movement, we're performing shoulder flexion, basically lifting your arm up overhead, which will be handled by the anterior or front deltoid and, to a lesser degree, the clavicular or upper head of the pecs. We'll also be performing elbow extension, which will hit all three heads of the triceps, and when viewed from the back, you can see that there will be scapular upward rotation occurring, handled by the upper traps. I like the overhead press for two main reasons. First, being a basic multi-joint barbell movement, it allows for a good deal 

#### Jeff Nippard: Bench Press transcript

In [5]:
jeff_bench = "/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/processed/cleaned_nippard_bench_dict"
jeff_bench_chunked = split_text_add_video_metadata(jeff_bench)
print("Chunks:", len(jeff_bench_chunked))
print("Sample chunk:", jeff_bench_chunked[:1])

Chunks: 16
Sample chunk: [Document(metadata={'video_id': 'vcBig73ojpE', 'title': 'How To Get A Huge Bench Press with Perfect Technique', 'author': 'Jeff Nippard', 'difficulty': 'intermediate', 'exercise_type': 'bench_press'}, page_content="Welcome, everyone, to the first episode of Technique Tuesday, where every week we're going to take an in-depth look at the lost art and science of training technique. Just as a quick general outline, for the most part, we will break each exercise into four sections. We will look at the muscles we'll be targeting, how to set up for the exercise, the execution of the movement, and common errors that I see many people make. So without further ado, let's jump right into it with the bench press exercise.")]


#### Jeff Nippard: Squat transcript

In [6]:
jeff_squat = "/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/processed/cleaned_nippard_squat_dict"
jeff_squat_chunked = split_text_add_video_metadata(jeff_squat)
print("Chunks:", len(jeff_squat_chunked))
print("Sample chunk:", jeff_squat_chunked[:1])

Chunks: 19
Sample chunk: [Document(metadata={'video_id': 'bEv6CCg2BC8', 'title': 'How To Get A Huge Squat With Perfect Technique (Fix Mistakes)', 'author': 'Jeff Nippard', 'difficulty': 'intermediate', 'exercise_type': 'squat'}, page_content="Okay, welcome everyone to a new episode of Technique Tuesday. This week, we're going to be looking at how to perform the squat with perfect technique. The back squat is often referred to as the king of lower body exercises and even the king of all exercises, and that's an appointment I think it actually deserves. Before we jump into the technique for this exercise, let's take a look at what muscles we're going to be targeting with this movement first.")]



#### Learning to Bench Press | The Starting Strength Method transcript

In [7]:
strength_bench = "/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/processed/cleaned_starting_strength_bench_dict"
strength_bench_chunked = split_text_add_video_metadata(strength_bench)
print("Chunks:", len(strength_bench_chunked))
print("Sample chunk:", strength_bench_chunked[:1])

Chunks: 6
Sample chunk: [Document(metadata={'video_id': 'rxD321l2svE', 'title': 'Learning to Bench Press | The Starting Strength Method', 'author': 'Mark Rippetoe', 'difficulty': 'advanced', 'exercise_type': 'bench_Press'}, page_content="When you're learning how to bench press, it might be prudent to use a spotter if one is available. However, if you're working inside a correctly set up power rack, a spotter is not absolutely necessary. Start with an empty bar. Lie down on the bench with your eyes looking straight up. In this position, you should be far enough down from the bar—meaning toward the foot end of the bench—that when you look up, your eyes are focused on the downside of the bar. Your feet should be flat on the ground with your shins approximately vertical. Your upper back should be flat against the bench with your lower back in an anatomically normal arched position. Take an overhand grip on the bar; your grip should be somewhere between 22 and 24 inches measured between the

#### How To Bench Press: Layne Norton's Complete Guide transcript



In [8]:
bodybuilding_bench = "/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/processed/cleaned_bodybuilding_bench_dict"
bodybuilding_bench_chunked = split_text_add_video_metadata(bodybuilding_bench)
print("Chunks:", len(bodybuilding_bench_chunked))
print("Sample chunk:", bodybuilding_bench_chunked[:1])

Chunks: 18
Sample chunk: [Document(metadata={'video_id': 'esQi683XR44', 'title': "How To Bench Press: Layne Norton's Complete Guide", 'author': 'Layne Norton', 'difficulty': 'beginner', 'exercise_type': 'bench_press'}, page_content="Research is my passion. Muscle and strength are my pursuits. I'm the powerlifter, bodybuilder, scientist, and coach. I'm Layne Norton. I am a physique architect. The bench press is one of the most important upper body exercises. This makes it crucial for overall development and strength. A lot of people think about it as just a chest movement, but it incorporates the pectorals, the triceps, the shoulders, and even the back when done correctly. When it's done wrong, the results can be disastrous. Seven years ago, I tore my right pectoral bench pressing incorrectly. But over time, I learned to incorporate more muscle groups and focus on the movement itself. Not only did it become more safe, but my results were better. In this video, I'm going to teach you thi

#### How To: Incline Barbell Bench Press | 3 GOLDEN RULES

In [9]:
incline_bench = "/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/processed/cleaned_incline_bench_dict.json"
incline_bench_chunked = split_text_add_video_metadata(incline_bench)
print("Chunks:", len(incline_bench_chunked))
print("Sample chunk:", incline_bench_chunked[:1])

Chunks: 6
Sample chunk: [Document(metadata={'video_id': 'SrqOu55lrYU', 'title': 'How To: Incline Barbell Bench Press | 3 GOLDEN RULES! (MADE BETTER!)', 'author': 'ScottHermanFitnesss', 'difficulty': 'beginner', 'exercise_type': 'incline_bench_press'}, page_content='What’s going on, Nation? I’m Scott from MuscularStrength.com, and today we’re going to go over the three golden rules for performing a barbell incline bench press. But before we get started, if you’ve been enjoying my Golden Rule series, be sure to smash that like and subscribe button, and let me know which exercise you want to see next down in the comment section below. Alright, guys, golden rule number one: you always want to make sure you’re benching with a grip just outside shoulder width. I know some of you are going to say that with a wider grip you can bench more weight, but when you have a wider grip on this exercise using a barbell, you’re not going to get much of a squeeze at the top of the movement. It should go w

#### Embed chunks as vectors into ChromaDB using OpenAI embedding model

In [10]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings 

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
#  embeddings tells Chroma how to turn your text into vectors. 
# You're passing it in so Chroma can embed each document's page_content before storing it.

all_docs = jeff_ohp_chunked + jeff_bench_chunked + jeff_squat_chunked + strength_bench_chunked + bodybuilding_bench_chunked + incline_bench_chunked


vectorstore = Chroma.from_documents(all_docs, embeddings, persist_directory="/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/chroma_db")
# index the documents and embedddings with a persistent directory in chroma db

print(vectorstore._collection.count())  # Should show your doc count now

76


# Pipeline: user query, image classification, image analysis, conversation memory response
- Still to add: query router

In [ ]:
def user_input(user_query, user_video, session_id):
    if user_video:
        user_video_encoded = analyze_video(filepath_in=user_video, frame_count=15, max_seconds=10) # encodes video
        encoded_images = [] # creates a list of encoded images for LLM
        for images in user_video_encoded:
            encoded_images.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{images}"}})
            classification_image = user_video_encoded[0] 

        router_llm = ChatOpenAI(model='gpt-4o')

        response = router_llm.invoke([

        {"role": "user", "content": 
        
        [{"type": "text", "text":  """Your job is to analyze images of users working out for proper form, and list the key checkpoints of their to body evaluate. 
        Give me ONLY the bodypart checkpoints. Do NOT include evaluation suggestions. Do NOT include an intro sentence. 
        Output format should be exactly the example below.
        **Example**
        Overhead press

        1. Feet & base
        2. Glutes & legs
        3. Core & Ribcage
        4. Shoulder position
        5. Bar path
        6. Head & Neck
        7. Lockout position
        8. Tempo and control
        """},


            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{classification_image}"}}
            ]}


        ])
        print(response.text)

        cleaned = clean_classification_text(response)
        print(cleaned)

        
        retrieval_query = cleaned

        results_user = vectorstore.similarity_search(user_query, k=5)
        results_video = vectorstore.similarity_search(retrieval_query, k=5)
        results = results_user + results_video

        unique_results = []
        seen = set()

        for r in results:
            content = r.page_content
            if content not in seen:
                seen.add(content)
                unique_results.append(r)


        context = "\n".join([r.page_content for r in unique_results])

        prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a world-class fitness coach. You have extensive experience in helping weight lifters achieve perfect form and maximum hypertrophy. 
        Your job is to analyze images of users lifting weights, offer them advice from your context, and to answer any questions they might have. 
        Inspect each image CLOSELY and arefuly for problems or issues related to best practices in exercise form. Help the user diagnose their incorrect form. 
        Be specific about what you observe.

        # ANSWER CONTEXT
        Use ONLY the following context when answering a user: 
            
        ---   
        {context}
        ---
        if the query or image isn't in context, reply, 'I don't have expert coaching content for this exercise yet. 
        Currently I can analyze: bench press, overhead press, incline bench press...'"
        """),
            MessagesPlaceholder(variable_name="history"),
            MessagesPlaceholder(variable_name="input"),
        ])

        user_msg = HumanMessage(content=[
            {"type": "text", "text": user_query},
            *encoded_images,   # <- your list of {"type":"image_url",...}
        ])

        llm = ChatOpenAI(model='gpt-5',
                        temperature=0.5)

        output_parser = StrOutputParser()

        chain = prompt | llm | output_parser

        chain_with_history = RunnableWithMessageHistory(
            chain,
            get_session_history=get_chat_history,
            input_messages_key="input",
            history_messages_key="history"

        )

        response = chain_with_history.invoke(
            {"input": [user_msg], "context": context},
            config={"configurable": {"session_id": session_id}}
        )

        print(textwrap.fill(response))

        print(response)
        return response

    if user_query:
        results_user = vectorstore.similarity_search(user_query, k=5)
            


user_query = "how's my form?"
user_video = "/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/raw_workout_videos/user_01_bench.MP4"
user_input(user_video=user_video, user_query=user_query, session_id=123)

Frames processed: 15, (10s cap)
Saved to processed-images/user_01_bench
Video name: user_01_bench
Bench Press

1. Feet & base
2. Lower back & arch
3. Glutes & hips
4. Shoulder position
5. Hand placement
6. Bar path
7. Head & Neck
8. Elbow position
9. Tempo and control
Bench Press     Feet   base    Lower back   arch    Glutes   hips    Shoulder position    Hand placement    Bar path    Head   Neck    Elbow position    Tempo and control
Thanks for the clips—nice job setting a big arch and getting your
chest high. A few key fixes will make this much stronger and safer.
What I’m seeing - Your glutes are coming off the bench during the set.
You want four points of contact the whole time: head, upper back,
glutes, and feet all planted. If you unrack by yourself, it’s fine to
keep the butt up for the liftoff, but drop the hips before the first
rep and keep them down. - Your feet are pulled very far back and
you’re up on your toes, which is why the hips are bridging. This also
makes leg drive

'Thanks for the clips—nice job setting a big arch and getting your chest high. A few key fixes will make this much stronger and safer.\n\nWhat I’m seeing\n- Your glutes are coming off the bench during the set. You want four points of contact the whole time: head, upper back, glutes, and feet all planted. If you unrack by yourself, it’s fine to keep the butt up for the liftoff, but drop the hips before the first rep and keep them down.\n- Your feet are pulled very far back and you’re up on your toes, which is why the hips are bridging. This also makes leg drive inefficient.\n\nHow to fix it (step by step)\n- Set your eyes directly under the bar before the unrack. Unrack by lifting out, not up, then set your hips down.\n- Plant the feet flat. Move them slightly forward until your heels can stay on the floor and your shins are nearly vertical. You can point the toes out a bit to help keep the heels down and keep the legs close to the bench when viewed from the front.\n- Drive through the 

In [ ]:
# print(len(results_user))
# print(len(results_video))
# print(len(results))
# print(len(unique_results))

5
5
10
10


In [ ]:
# for r in results:
#     print(r.metadata)

{'title': "How To Bench Press: Layne Norton's Complete Guide", 'video_id': 'esQi683XR44', 'difficulty': 'beginner', 'exercise_type': 'bench_press', 'author': 'Layne Norton'}
{'title': 'How To: Incline Barbell Bench Press | 3 GOLDEN RULES! (MADE BETTER!)', 'exercise_type': 'incline_bench_press', 'difficulty': 'beginner', 'author': 'ScottHermanFitnesss', 'video_id': 'SrqOu55lrYU'}
{'author': 'ScottHermanFitnesss', 'exercise_type': 'incline_bench_press', 'title': 'How To: Incline Barbell Bench Press | 3 GOLDEN RULES! (MADE BETTER!)', 'video_id': 'SrqOu55lrYU', 'difficulty': 'beginner'}
{'title': 'How To Get A Huge Bench Press with Perfect Technique', 'author': 'Jeff Nippard', 'difficulty': 'intermediate', 'video_id': 'vcBig73ojpE', 'exercise_type': 'bench_press'}
{'title': 'Build Bigger Shoulders With Perfect Training Technique (The Overhead Press)', 'author': 'Jeff Nippard', 'exercise_type': 'overhead_press', 'video_id': '_RlRDWO2jfg', 'difficulty': 'intermediate'}
{'video_id': 'rxD321l2

In [ ]:
# # vectorstore cosine search: use if you need to verify GPT results
# unique_results = vectorstore.similarity_search(retrieval_query, k=10)

# for i, doc in enumerate(unique_results, 1):
#     print(f"==Results {i}==")
#     print(textwrap.fill(doc.page_content, width=80))
#     print(doc.metadata["exercise_type"])
#     print("\n")

==Results 1==
at the ceiling, unlock your elbows, lower the bar to your chest, touch without
stopping, and then drive the bar back at the point on the ceiling your eyes have
trapped. Try it for a set of five reps. You'll immediately notice that if your
eyes don't move from their fixed position, the bar will go to the same place
every rep. Do another set of five with the bar, reinforcing your eye position,
and then rack the bar. This is done with locked elbows after the last rep is
finished. Should you have a spotter, this movement back to the rack should be
covered. Now that we've covered the basic movement pattern for the bench press,
we can maximize the efficiency of the lift by improving the position of the
upper back, legs, and feet. The upper back needs to be planted firmly against
the bench and used as a platform to drive against while the arms drive the bar
up. When this is done correctly, the shoulder blades will be adducted or pinched
together. Sit on the bench in the same pos